In [ ]:
"""Main execution script for session-level conversation experiments"""

from langfuse import get_client, propagate_attributes
from langfuse.openai import OpenAI
from agents.chatbot import LlamaChatbot
from agents.simulator import SimulatedUser
from metrics.metrics import EvaluationMetrics
from utils.utils import (
    load_config, 
    validate_langfuse_connection, 
    SessionManager,
    logger
)
from typing import Dict, List
import json

# Load configuration
config = load_config()

# Initialize Langfuse client
langfuse = get_client()

# Verify connection
if not validate_langfuse_connection(langfuse):
    raise RuntimeError("❌ Langfuse authentication failed")

logger.info("✅ Langfuse client is authenticated and ready!")

# Initialize evaluation metrics
metrics = EvaluationMetrics(
    judge_model=config['models']['judge_model'],
    ollama_config=config['ollama']
)

def simulate_single_turn(
    chatbot: LlamaChatbot,
    simulated_user: SimulatedUser,
    turn_number: int,
    session_id: str,
    persona: str,
    scenario: str,
    is_first: bool = False
) -> Dict:
    """
    Simulate a single turn as a separate trace within the session
    
    Args:
        chatbot: LlamaChatbot instance
        simulated_user: SimulatedUser instance
        turn_number: Current turn number
        session_id: Session identifier for grouping traces
        persona: User persona description
        scenario: Conversation scenario
        is_first: Whether this is the first turn
        
    Returns:
        Dict containing turn data with trace_id
    """
    # Generate deterministic trace ID for this turn
    trace_id = SessionManager.generate_trace_id(langfuse, session_id, turn_number)
    
    # Start a new observation for this turn
    with langfuse.start_as_current_observation(
        as_type="span",
        name=f"Turn {turn_number}",
        trace_context={"trace_id": trace_id}
    ) as span:
        # Propagate session_id to all child observations
        with propagate_attributes(session_id=session_id):
            # User generates message
            user_message = simulated_user.generate_message(is_first_turn=is_first)
            
            # Assistant responds
            assistant_response = chatbot.chat(user_message, turn_number)
            
            # Update simulated user's memory
            simulated_user.update_history(user_message, assistant_response)
            
            # Update the trace with turn information
            langfuse.update_current_trace(
                name=f"Turn {turn_number}",
                input={"user_message": user_message},
                output={"assistant_response": assistant_response},
                metadata={
                    "turn_number": turn_number,
                    "persona": persona,
                    "scenario": scenario
                }
            )
            
            return {
                "turn": turn_number,
                "user": user_message,
                "assistant": assistant_response,
                "trace_id": trace_id
            }

def simulate_continuous_conversation(
    persona: str,
    scenario: str,
    session_id: str,
    max_turns: int = 6
) -> Dict:
    """
    Simulate a continuous conversation with each turn as a separate trace
    All traces are grouped under the same session_id
    
    Args:
        persona: User persona description
        scenario: Conversation scenario
        session_id: Session identifier for grouping all traces
        max_turns: Maximum number of conversation turns
        
    Returns:
        Dict containing conversation log, session_id, and metadata
    """
    # Initialize chatbot with session tracking
    chatbot = LlamaChatbot(
        model=config['models']['answer_model'],
        ollama_config=config['ollama'],
        session_id=session_id
    )
    
    # Initialize simulated user
    simulated_user = SimulatedUser(
        persona=persona,
        scenario=scenario,
        model=config['models']['answer_model_alt'],
        ollama_config=config['ollama']
    )
    
    conversation_log = []
    
    # Log session start
    SessionManager.log_session_start(session_id, persona, scenario)
    
    # Simulate each turn
    for turn in range(1, max_turns + 1):
        logger.info(f"--- Turn {turn} ---")
        
        is_first = (turn == 1)
        turn_result = simulate_single_turn(
            chatbot,
            simulated_user,
            turn,
            session_id,
            persona,
            scenario,
            is_first
        )
        
        conversation_log.append(turn_result)
        
        logger.info(f"USER: {turn_result['user'][:100]}...")
        logger.info(f"ASSISTANT: {turn_result['assistant'][:100]}...")
        logger.info("")
    
    return {
        "conversation_log": conversation_log,
        "session_id": session_id,
        "num_turns": len(conversation_log)
    }

def run_task(*, item, **kwargs) -> Dict:
    """
    Task function for Langfuse experiment with session-level evaluation
    This function is called by dataset.run_experiment() for each dataset item
    
    Args:
        item: Dataset item containing input (persona, scenario)
        **kwargs: Additional arguments from run_experiment
        
    Returns:
        Dict containing session data - this output is passed to evaluators
    """
    persona = item.input.get("persona", "")
    scenario = item.input.get("scenario", "")
    
    # Generate unique session ID for this conversation
    session_id = SessionManager.generate_session_id(item.id, prefix="session-eval")
    
    logger.info(f"\n🔄 Starting continuous multi-turn conversation...")
    
    # Simulate the full conversation
    result = simulate_continuous_conversation(
        persona=persona,
        scenario=scenario,
        session_id=session_id,
        max_turns=config['dataset']['max_turns']
    )
    
    logger.info(f"✅ Completed {result['num_turns']} turns under session: {session_id}")
    
    # Log session completion
    SessionManager.log_session_complete(
        session_id,
        result['num_turns'],
        {}
    )
    
    logger.info(f"{'='*80}\n")
    
    # Return structure for evaluators
    return {
        "session_id": session_id,
        "num_turns": result["num_turns"],
        "conversation_log": result["conversation_log"]
    }

if __name__ == "__main__":
    # Load dataset
    dataset_name = config['dataset']['name']
    dataset = langfuse.get_dataset(dataset_name)
    
    logger.info(f"📚 Fetching dataset '{dataset_name}' from Langfuse...")
    logger.info(f"Found {len(dataset.items)} items.\n")
    
    # Run experiment with session-level evaluators
    logger.info("🚀 Starting experiment with session-level evaluation...")
    
    result = dataset.run_experiment(
        name="session-level-evaluation-v1",
        description="Multi-turn conversations with session-level evaluation",
        task=run_task,
        evaluators=[
            metrics.create_session_relevance_evaluator(),
            metrics.create_session_accuracy_evaluator(),
            metrics.create_session_helpfulness_evaluator(),
            metrics.create_session_clarity_evaluator()
        ],
        run_evaluators=[
            metrics.create_session_quality_run_evaluator()
        ]
    )
    
    # Flush all events to Langfuse
    langfuse.flush()
    
    # Log completion summary
    logger.info("\n" + "="*80)
    logger.info("✅ EXPERIMENT COMPLETE!")
    logger.info("="*80)
    logger.info(f"✓ Each turn is stored as a separate trace")
    logger.info(f"✓ All turns grouped under their session IDs")
    logger.info(f"✓ Session-level evaluation scores attached via evaluators")
    logger.info(f"✓ Run-level aggregate metrics calculated")
    logger.info(f"✓ {len(dataset.items)} conversations evaluated")
    logger.info(f"✓ View results in Langfuse UI under Dataset Runs")
    logger.info("="*80)